<a href="https://colab.research.google.com/github/rihaneh813-svg/DataScience_project/blob/main/project_data_science_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("hello")

hello


***Classification Classic***


In [8]:
# ==========================
# 1. IMPORT LIBRARIES
# ==========================

import pandas as pd
import numpy as np
import re

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import classification_report, accuracy_score

In [9]:
df = pd.read_csv(
    "Phishing_Email.csv",
    on_bad_lines="skip"
)
df.head()

,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [10]:
# ==========================
# 3. DROP USELESS COLUMN
# ==========================

if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)

In [11]:
# ==========================
# 4. HANDLE NULL VALUES
# ==========================

print("\nNull values:")
print(df.isnull().sum())

df.dropna(inplace=True)


Null values:
Email Text    16
Email Type     0
dtype: int64


In [12]:
# ==========================
# 5. REMOVE DUPLICATES
# ==========================

print("\nDuplicates:", df.duplicated().sum())

df.drop_duplicates(inplace=True)


Duplicates: 1096


In [13]:
# ==========================
# 6. CLEAN TEXT
# ==========================

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df['clean_text'] = df['Email Text'].apply(clean_text)



In [14]:
# ==========================
# 7. ENCODE LABELS
# ==========================

encoder = LabelEncoder()

df['label'] = encoder.fit_transform(df['Email Type'])

print("\nLabels:")
print(df[['Email Type', 'label']].drop_duplicates())




Labels:
       Email Type  label
0      Safe Email      1
3  Phishing Email      0


In [15]:

# ==========================
# 8. TF-IDF
# ==========================

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df['clean_text'])

y = df['label']

print("\nTF-IDF Shape:", X.shape)




TF-IDF Shape: (17538, 5000)


In [16]:
print(df['Email Type'].value_counts())

Email Type
Safe Email        10980
Phishing Email     6558
Name: count, dtype: int64


In [17]:
# ==========================
# 9. TRAIN / TEST SPLIT
# ==========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])



Training samples: 14030
Testing samples: 3508


In [18]:
# ==========================
# 10. LOGISTIC REGRESSION
# ==========================

lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("\n===== Logistic Regression =====")

print(classification_report(y_test, y_pred_lr))

print("Accuracy:",
      accuracy_score(y_test, y_pred_lr))




===== Logistic Regression =====
              precision    recall  f1-score   support

           0       0.96      0.95      0.96      1312
           1       0.97      0.98      0.98      2196

    accuracy                           0.97      3508
   macro avg       0.97      0.97      0.97      3508
weighted avg       0.97      0.97      0.97      3508

Accuracy: 0.9689281641961232


In [19]:
# ==========================
# 11. SVM
# ==========================

svm = LinearSVC()

svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)

print("\n===== SVM =====")

print(classification_report(y_test, y_pred_svm))

print("Accuracy:",
      accuracy_score(y_test, y_pred_svm))




===== SVM =====
              precision    recall  f1-score   support

           0       0.97      0.97      0.97      1312
           1       0.98      0.98      0.98      2196

    accuracy                           0.98      3508
   macro avg       0.97      0.97      0.97      3508
weighted avg       0.98      0.98      0.98      3508

Accuracy: 0.9754846066134549


In [20]:
# ==========================
# 12. NAIVE BAYES
# ==========================

nb = MultinomialNB()

nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

print("\n===== Naive Bayes =====")

print(classification_report(y_test, y_pred_nb))

print("Accuracy:",
      accuracy_score(y_test, y_pred_nb))


===== Naive Bayes =====
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      1312
           1       0.96      0.98      0.97      2196

    accuracy                           0.96      3508
   macro avg       0.96      0.96      0.96      3508
weighted avg       0.96      0.96      0.96      3508

Accuracy: 0.9626567844925884


***DistilBERT***


In [21]:
# ==========================================================
# 13. INSTALL REQUIRED LIBRARIES
# ==========================================================

!pip install -q transformers datasets accelerate

In [3]:
# ==========================================================
# 14. IMPORT DISTILBERT LIBRARIES
# ==========================================================

import numpy as np

from datasets import Dataset

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [22]:
# ==========================================================
# 15. TRAIN / TEST SPLIT FOR DISTILBERT
# ==========================================================

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["clean_text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Training samples:", len(train_texts))
print("Testing samples:", len(test_texts))

Training samples: 14030
Testing samples: 3508


In [6]:
# ==========================================================
# 16. LOAD DISTILBERT TOKENIZER
# ==========================================================

tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [23]:
# ==========================================================
# 17. TOKENIZE TEXT DATA
# ==========================================================

train_encodings = tokenizer(
    train_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    test_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

In [24]:
# ==========================================================
# 18. CONVERT DATA TO HUGGINGFACE DATASET FORMAT
# ==========================================================

train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": train_labels.tolist()
})

test_dataset = Dataset.from_dict({
    "input_ids": test_encodings["input_ids"],
    "attention_mask": test_encodings["attention_mask"],
    "labels": test_labels.tolist()
})

In [25]:
# ==========================================================
# 19. LOAD PRETRAINED DISTILBERT MODEL
# ==========================================================

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [26]:
# ==========================================================
# 20. DEFINE EVALUATION METRIC
# ==========================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy
    }

In [27]:
# ==========================================================
# 21. TRAINING CONFIGURATION
# ==========================================================

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    report_to="none"
)

In [28]:
# ==========================================================
# 22. CREATE TRAINER OBJECT
# ==========================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
# ==========================================================
# 23. TRAIN DISTILBERT MODEL
# ==========================================================

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
# ==========================================================
# 24. PREDICT ON TEST DATA
# ==========================================================

predictions = trainer.predict(
    test_dataset
)

y_pred_distilbert = np.argmax(
    predictions.predictions,
    axis=1
)

In [ ]:
# ==========================================================
# 25. DISTILBERT EVALUATION
# ==========================================================

print("\n===== DISTILBERT =====\n")

print(
    classification_report(
        test_labels,
        y_pred_distilbert
    )
)

distilbert_acc = accuracy_score(
    test_labels,
    y_pred_distilbert
)

print("Accuracy:", distilbert_acc)

In [ ]:
# ==========================================================
# 26. FINAL COMPARISON OF ALL MODELS
# ==========================================================

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "SVM",
        "Naive Bayes",
        "DistilBERT"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_svm),
        accuracy_score(y_test, y_pred_nb),
        distilbert_acc
    ]
})

comparison = comparison.sort_values(
    by="Accuracy",
    ascending=False
)

print("\n===== FINAL COMPARISON =====\n")

print(comparison)